# 08 — Synthesis and community products

Pull the run manifests together into the maps and summaries the working group actually shares. Subject to the §11 data-sovereignty sign-off.

**Reads** manifests in `runs/`, products from `03`–`07`  
**Writes** community-facing maps, summary tables  
**Status** Phase 7 — skeleton, gated on §11

> Skeleton. Section headings and the config cell are in place; the analysis cells are deliberately empty for the group to fill in together.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio

def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another tile or another river is a single-cell edit.

In [ ]:
# ---- What to synthesize ----
TILE = "13TFJ"
RUNS = {
    "labels":    "labels_smoketest_redshirt_06403700_2022",
    "phenology": "phenology_vbet_13TFJ",
    "extent":    "extent_phenology_vbet_13TFJ",
    "condition": "condition_phenology_vbet_13TFJ",
    "drivers":   "drivers_13TFJ",
}

# ---- Release gate ----
# Plan 11 is an OPEN decision. This repo is public and a 60 cm gallery map is exactly the
# kind of product that needs Tribal sign-off before it leaves the group. Keep this False
# until that decision is recorded in the plan.
CARE_SIGNOFF = False

# ---- Outputs ----
RUN_NAME = f"synthesis_{TILE}"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME

## 2. Load the manifests

Every number in a product should trace to a manifest in `runs/`. If it doesn't, it isn't reproducible and shouldn't ship.

## 3. Corridor summary

Gallery extent and condition by reach, with uncertainty carried through from `05`/`06`.

## 4. Maps

Reference maps use Esri, not CARTO — CARTO now burns an "API KEY REQUIRED" watermark into tiles while still returning HTTP 200, so it fails silently.

## 5. Export

Gated on `CARE_SIGNOFF`. Until that is True this section writes nothing outside `data/`.

## 6. Save and record the run

Every output gets a manifest in `runs/` — small, text, always committed, even when the raster it describes is not.

In [ ]:
manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "08_Synthesis_and_Products.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs":      {},          # STAC item IDs, upstream run names, source manifests
    "parameters":  {},          # everything from the config cell
    "environment": {"python": sys.version.split()[0]},
    "results":     {},
    "outputs":     [],
}

# manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
# manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")

## What comes next

Phase 6 is the real next step for the science: run the frozen workflow on 13TGK as a transferability test before the remaining five squares.